In [91]:
import pandas as pd
import numpy as np

melb_df = pd.read_csv('data/melb_df_tuned2.csv', sep=';')
melb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  int64  
 10  Bedroom        13580 non-null  int64  
 11  Bathroom       13580 non-null  int64  
 12  Car            13580 non-null  int64  
 13  Landsize       13580 non-null  float64
 14  BuildingArea   13580 non-null  float64
 15  CouncilArea    12211 non-null  str    
 16  Lattitude      13580 non-null  float64
 17  Longtitude     13580 non-null  float64
 18  Regionname     13

In [92]:
# Адрес -- весьма уникальная штука
melb_df['Address'].nunique()

13378

In [93]:
print(melb_df['Address'].loc[177])
print(melb_df['Address'].loc[1812])
print(melb_df['Address'].loc[9001])
print(melb_df['Address'].loc[302])

2/119 Railway St N
9/400 Dandenong Rd
172 Danks St
2/78 Ormond Rd


In [94]:
def unifySubType(str_name):
    shorter = {
        'Avenue': 'Av',
        'Boulevard': 'Bvd',
        'Parade': 'Pde'
    }
    if str_name in shorter:
        return shorter[str_name]
    else:
        return str_name

In [95]:
def getStreetType(str_name):
    '''
    Итак, адрес строится следующим образом:
    сначала указывается номер дома и корпус, после указывается название улицы,
    а в конце — подтип улицы,
    но в некоторых случаях к подтипу добавляется географическая отметка (N — север, S — юг и т. д.), она нам не нужна
    '''

    list_name = str_name.split(' ')
    return list_name[-2] if list_name[-1] in {'N', 'S', 'W', 'E'} else list_name[-1]
    

In [96]:
street_types = melb_df['Address'].apply(getStreetType).apply(unifySubType)
print(street_types.nunique())
street_types.value_counts(normalize=True)

53


Address
St           0.589985
Rd           0.208027
Ct           0.045066
Dr           0.032916
Av           0.026583
Gr           0.022901
Pde          0.016642
Pl           0.012445
Cr           0.011193
Cl           0.007364
La           0.004934
Bvd          0.004860
Tce          0.003461
Wy           0.002946
Cct          0.001841
Hwy          0.001767
Sq           0.000810
Crescent     0.000663
Cir          0.000515
Strand       0.000515
Esplanade    0.000442
Grove        0.000368
Mews         0.000295
Gdns         0.000295
Fairway      0.000295
Grn          0.000295
Righi        0.000221
Crossway     0.000221
Esp          0.000147
Crofts       0.000147
Victoria     0.000147
Ridge        0.000147
Highway      0.000074
Athol        0.000074
Grange       0.000074
Res          0.000074
Cove         0.000074
Qy           0.000074
Glade        0.000074
Nook         0.000074
Terrace      0.000074
Loop         0.000074
East         0.000074
Dell         0.000074
Eyrie        0.000074
Gr

Из данного вывода можно увидеть, что есть группа наиболее популярных подтипов улиц, а дальше частота подтипов быстро падает.

В таком случае давайте применим очень распространённый метод уменьшения количества уникальных категорий — выделим n подтипов, которые встречаются чаще всего, а остальные обозначим как 'other' (другие).

In [97]:
#отберём 10 наиболене популярных подтипов улиц
common_street_types = street_types.value_counts().nlargest(10).index.tolist()
common_street_types = set(common_street_types)
display(common_street_types)

{'Av', 'Cl', 'Cr', 'Ct', 'Dr', 'Gr', 'Pde', 'Pl', 'Rd', 'St'}

In [98]:
# Определим новый столлбец?
melb_df['StreetType'] = street_types.apply(
    lambda x: x if x in common_street_types else 'other'
)
melb_df['StreetType'].value_counts()

StreetType
St       8012
Rd       2825
Ct        612
Dr        447
other     365
Av        361
Gr        311
Pde       226
Pl        169
Cr        152
Cl        100
Name: count, dtype: int64

Теперь, у нас нет потребности хранить признак Address, так как, если конкретное местоположение объекта всё же и влияет на его стоимость, то оно определяется столбцами Longitude и Lattitude. Удалим его из нашей таблицы:

In [99]:
melb_df = melb_df.drop('Address', axis=1)

Примените эту функцию к столбцу и вычислите среднюю цену объекта недвижимости, проданного в выходные дни. Результат округлите до целых.

In [100]:

melb_df['WeekdaySale'].value_counts(normalize=True)

WeekdaySale
5    0.865906
6    0.078277
0    0.043962
1    0.009794
3    0.002062
Name: proportion, dtype: float64

In [101]:
def getWeekend(val):
    return 1 if val in [5, 6] else 0
melb_df['Weekend'] = melb_df['WeekdaySale'].apply(getWeekend)


In [102]:
# средняя цена объекта, проданного в выходные
round(melb_df[melb_df['Weekend'] == 1]['Price'].mean(), 0)

np.float64(1081199.0)

In [103]:
sellergcommon = melb_df['SellerG'].value_counts().nlargest(49).index.tolist()
sellergcommon = set(sellergcommon)
sellergcommon

melb_df['SellerG'] = melb_df['SellerG'].apply(
    lambda x: x if x in sellergcommon else 'other'
)
display(melb_df['SellerG'].value_counts(normalize=True))

SellerG
Nelson           0.115243
Jellis           0.096907
other            0.088292
hockingstuart    0.085935
Barry            0.074448
Ray              0.051620
Marshall         0.048527
Buxton           0.046539
Biggin           0.028940
Brad             0.025184
Fletchers        0.022165
Woodards         0.022165
Jas              0.017894
Greg             0.017599
McGrath          0.016348
Sweeney          0.015906
Noel             0.015096
Miles            0.014433
RT               0.013549
Gary             0.012518
Harcourts        0.012371
Hodges           0.011561
YPA              0.011340
Stockdale        0.011046
Village          0.009205
Kay              0.008763
Raine            0.008542
Williams         0.008174
Love             0.008027
Douglas          0.007143
Chisholm         0.005670
RW               0.005155
Rendina          0.004860
HAR              0.004566
O'Brien          0.004492
C21              0.004197
Collins          0.004124
Cayzer           0.003829
Purp

во сколько раз минимальная цена объектов недвижимости, проданных компанией 'Nelson', больше минимальной цены объектов, проданных компаниями, обозначенными как 'other'

In [104]:
Nelson_min_price = melb_df[melb_df['SellerG'] == 'Nelson']['Price'].min()
Other_min_price = melb_df[melb_df['SellerG'] == 'other']['Price'].min()
round(Nelson_min_price/Other_min_price, 1)


np.float64(1.3)

In [105]:
melb_df.info()
melb_df.to_csv('data/melb_df_tuned3.csv', index=False, sep=';')

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 26 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Rooms          13580 non-null  int64  
 2   Type           13580 non-null  str    
 3   Price          13580 non-null  float64
 4   Method         13580 non-null  str    
 5   SellerG        13580 non-null  str    
 6   Date           13580 non-null  str    
 7   Distance       13580 non-null  float64
 8   Postcode       13580 non-null  int64  
 9   Bedroom        13580 non-null  int64  
 10  Bathroom       13580 non-null  int64  
 11  Car            13580 non-null  int64  
 12  Landsize       13580 non-null  float64
 13  BuildingArea   13580 non-null  float64
 14  CouncilArea    12211 non-null  str    
 15  Lattitude      13580 non-null  float64
 16  Longtitude     13580 non-null  float64
 17  Regionname     13580 non-null  str    
 18  Propertycount  13